In [1]:
import csv
import importlib
import json
import os
import random
import sys
import torch
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from time import sleep
from collections import deque, defaultdict
from itertools import count
from typing import Any, Dict, Counter, List

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

from importnb import Notebook
with Notebook():
    from Labs.LatencyModel import LatencyModel, MultiDULatencyModel
    from Labs.Policy import DrlPolicy
    from Labs.CacheEngine import CacheEngineEnv
    from Labs.UserRequest import UserRequestEvents
    from Labs.EnvWrapper import EnvWrapper

from RL.Networks import QNetwork, MultiHeadQNetwork
from RL.Buffers import ReplayBuffer, NStepReplayBuffer
from RL.Adapters import FeatureAdapter, NetworkAdapter
from RL.FocusWorkers import BaseWorker, EnhWorker, FocusWorker
from RL.A2CWorker import A2CWorker

import Common.config as config
import Common.datatypes as datatypes
import Common.debugger as debugger
import Common.utils as utils
import Core.builders as builders

importlib.reload(builders)
importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(debugger)
importlib.reload(utils)

<module 'Common.utils' from 'c:\\Users\\es25591\\Workspace\\CacheVideoPredict360\\Sources\\Common\\utils.py'>

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

UserTransition = datatypes.UserTransition
CachePolicy = datatypes.CachePolicy
CacheKey = datatypes.CacheKey

# Note: The state and action dimensions are determined by the cache size and the specific 
# design of the state representation and action space. Adjust these calculations based on 
# your actual implementation of the state and action spaces.
cfg = config.Config()
cfg.filename = \
    f"focus_eps{cfg.epsilon_start}_" \
    f"lrdecay{cfg.learning_rate_decay}_" \
    f"gamma{cfg.gamma}.csv"
cfg.state_dim_base_focus = cfg.cache_size * 10 + 2
cfg.state_dim_enh_focus = cfg.cache_size * 10 + 2
cfg.action_dim_base_focus = cfg.cache_size * 5 + 1

debugger = debugger.debug

In [ ]:
class NetworkAdapter:
    def __init__(self, cfg: Any, env: Any, feature_adapter: Any):
        self.env = env
        self.cfg = cfg
        self.features = feature_adapter

        self.C = self.cfg.cache_size  # paper's cache capacity (videos)
        self.k = self.cfg.viewport    # paper's tiles per video (enhancement)

    def build_observation(self, idx_vp, video, tile = None) -> np.ndarray:

        cache = self.env.mec_cache.policy.cache

        x_s = np.zeros(len(cache), dtype=np.float32)
        x_l = np.zeros(len(cache), dtype=np.float32)

        for idx, (v, t) in enumerate(cache):
            if v == -1:
                continue

            if t == -1:
                x_s[idx] = self.features.video_freq_short.get(v, 0) / self.features.video_hist_short.maxlen
                x_l[idx] = self.features.video_freq_long.get(v, 0) / self.features.video_hist_long.maxlen
            else:
                x_s[idx] = self.features.tile_freq_short.get((v, t), 0) / self.features.tile_hist_short.maxlen
                x_l[idx] = self.features.tile_freq_long.get((v, t), 0) / self.features.tile_hist_long.maxlen

        if tile is None:
            y_s = np.array(
                [self.features.video_freq_short.get(video, 0) / self.features.video_hist_short.maxlen], 
                dtype=np.float32
            )
            y_l = np.array(
                [self.features.video_freq_long.get(video, 0) / self.features.video_hist_long.maxlen], 
                dtype=np.float32
            )
        else:
            y_s = np.array(
                [self.features.tile_freq_short.get((video, tile), 0) / self.features.tile_hist_short.maxlen], 
                dtype=np.float32
            )
            y_l = np.array(
                [self.features.tile_freq_long.get((video, tile), 0) / self.features.tile_hist_long.maxlen], 
                dtype=np.float32
            )

        step_one_hot = int(tile is None)

        return np.concatenate(
            [x_s, x_l, y_s, y_l], 
            axis=0
        )

    def reset(self):
        
        obs, info = self.env.reset()
        self.features.reset_history()

        return obs, info
    
    def env_is_done(self) -> bool:
        return self.env.users_env.all_users_done()

In [4]:
def _append_csv_row(csv_path: str, fieldnames: list[str], row: dict) -> None:
    write_header = not os.path.exists(csv_path)
    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row)


def save_episode_metrics(
    metrics_dir: str,
    ep: int,
    total_reward: float,
    cache_hits: int,
    cache_misses: int,
    agent,
):
    csv_path = os.path.join(metrics_dir, 'episode_metrics.csv')
    fieldnames = [
        'episode',
        'total_reward',
        'cache_hits',
        'cache_misses',
        'hit_rate',
        'epsilon',
        'lr',
    ]

    row = {
        'episode': ep,
        'total_reward': round(float(total_reward), 2),
        'cache_hits': cache_hits,
        'cache_misses': cache_misses,
        'hit_rate': float(cache_hits) / float(cache_hits + cache_misses + 1e-9),
        'epsilon': round(float(agent.epsilon), 6) if agent else None,
        'lr': float(agent.scheduler.get_last_lr()[0]) if agent else None,
    }
    _append_csv_row(csv_path, fieldnames, row)

def save_step_metrics(
    metrics_dir: str,
    episode: int,
    episode_step: int,
    global_step: int,
    reward: float,
    agent,
    train_metrics: dict | None,
):
    csv_path = os.path.join(metrics_dir, 'step_metrics.csv')
    fieldnames = [
        'episode',
        'episode_step',
        'global_step',
        'reward',
        'epsilon',
        'lr',
        'train_loss',
        'actor_loss',
        'critic_loss'
    ]

    row = {
        'episode': episode,
        'episode_step': episode_step,
        'global_step': global_step,
        'reward': float(reward),
        'epsilon': round(float(agent.epsilon), 6) if agent else None,
        'lr': float(agent.scheduler.get_last_lr()[0]) if agent else None,
        'train_loss': None,
        'actor_loss': None,
        'critic_loss': None
    }

    if train_metrics is not None:
        row.update({
            'train_loss': train_metrics.get('train_loss'),
            'actor_loss': train_metrics.get('actor_loss'),
            'critic_loss': train_metrics.get('critic_loss'),
    })

    _append_csv_row(csv_path, fieldnames, row)

def update_metrics(info: dict, reward: float) -> tuple[float, int, int, int, int]:
    enh_hits = info.get("enh_layer_hits", 0)
    base_hits = info.get("base_layer_hits", 0)
    enh_misses = info.get("enh_layer_misses", 0)
    base_misses = info.get("base_layer_misses", 0)

    return reward, base_hits, base_misses, enh_hits, enh_misses

In [5]:
entropy_floor = 0.2
boltzmann_tau = 5.0

print(f"Entropy floor: {entropy_floor} | Boltzmann tau: {boltzmann_tau}")

def _force_boltzmann_action(agent, state_vec, tau=1.5):
    """Resample action from a temperature-scaled policy when entropy is too low."""
    state_t = torch.tensor(state_vec, dtype=torch.float32).unsqueeze(0).to(agent.device)
    with torch.no_grad():
        value, action_probs = agent.network(state_t)
        action_probs = torch.clamp(action_probs, min=1e-8)
        logits = torch.log(action_probs)
        boltz_probs = F.softmax(logits / tau, dim=-1)
        dist = torch.distributions.Categorical(probs=boltz_probs)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        entropy = dist.entropy()
    return action.item(), value.squeeze(), log_prob.squeeze(0), entropy.squeeze(0)


def select_action(agent, req_state, env, net_adapter=None):

    if req_state is None:
        return None, np.zeros(5, dtype=np.int32), [None] * (1 + cfg.viewport)

    backhaul_usage = 0
    missing = env._missing_items(req_state)
    transition = [None] * (1 + cfg.viewport)

    if missing[0] == 1:
        state_base = net_adapter.build_observation(0, req_state["video"])
        action_base, value_base, prob_base, entropy_base = agent.select_action(state_base)

        if float(entropy_base) < entropy_floor:
            action_base, value_base, prob_base, entropy_base = _force_boltzmann_action(
                agent, state_base, tau=boltzmann_tau
            )
            debugger.log("boltzmann_base_forced", 1)
        else:
            debugger.log("boltzmann_base_forced", 0)

        transition[0] = {
            'state': state_base,
            'action': action_base,
            'value': value_base,
            'prob': prob_base,
            'entropy': entropy_base
        }

        message = {
            "video": req_state["video"],
            "tiles": [],
            "base_req_init": True,
            "action_idx": action_base,
        }
        env.prefetch_fn(env.mec_cache, message)

        if action_base != 0:
            backhaul_usage += 12 * env.mec_cache.tile_size_bytes[0]

        debugger.log("base_action", action_base)
        debugger.log("entropy_base", entropy_base)

    for idx, missing_item in enumerate(missing[1:]):
        if missing_item == 1:
            state_enh = net_adapter.build_observation(
                idx + 1,
                req_state["video"],
                req_state["viewport"][idx]
            )
            action_enh, value_enh, prob_enh, entropy_enh = agent.select_action(state_enh)

            if float(entropy_enh) < entropy_floor:
                action_enh, value_enh, prob_enh, entropy_enh = _force_boltzmann_action(
                    agent, state_enh, tau=boltzmann_tau
                )
                debugger.log(f"boltzmann_enh_{idx}_forced", 1)
            else:
                debugger.log(f"boltzmann_enh_{idx}_forced", 0)

            transition[idx + 1] = {
                'state': state_enh,
                'action': action_enh,
                'value': value_enh,
                'prob': prob_enh,
                'entropy': entropy_enh
            }

            message = {
                "video": req_state["video"],
                "tiles": [req_state["viewport"][idx]],
                "base_req_init": False,
                "action_idx": action_enh,
            }
            env.prefetch_fn(env.mec_cache, message)

            if action_enh != 0:
                backhaul_usage += env.mec_cache.tile_size_bytes[1]

            debugger.log(f"enh_{idx}_action", action_enh)
            debugger.log(f"entropy_enh_{idx}", entropy_enh)

    debugger.log("backhaul_usage", backhaul_usage)

    debugger.log("base_layer_miss", missing[0])
    for i, is_missing in enumerate(missing[1:], start=1):
        debugger.log(f"enh_layer_missing_{i}", is_missing)

    return None, missing, transition


def run_drl_warmup(env, agent, net_adapter, cfg, warmup_steps=1000):
    """Populate cache using DRL-selected actions before the real scenario starts."""
    _, info = net_adapter.reset()
    effective_steps = int(getattr(cfg, "warmup_steps", warmup_steps))

    for warm_step in range(effective_steps):
        req_state = info.get("user_request", None)
        action, _, _ = select_action(agent, req_state, env, net_adapter)

        _, _, done, info = env.step(action, req_state, net_adapter)
        debugger.log("warmup_step", warm_step)

        if done or net_adapter.env_is_done():
            break

    return info

def run_episode(episode, env, agent, net_adapter, cfg, metrics_dir, global_step_start):
    """Run one full training episode and persist step-level metrics."""
    _, info = net_adapter.reset()

    total_reward = 0.0
    cache_hits = cache_misses = 0
    base_hits = base_misses = 0
    enh_hits = enh_misses = 0
    psnr_sum = 0.0

    global_step = global_step_start

    if cfg.has_warmup:
        info = run_drl_warmup(env, agent, net_adapter, cfg, warmup_steps=1000)

    if episode == 0:
        max_steps = 20000
    else:
        max_steps = 20000

    for step in range(max_steps):
        global_step += 1

        # --- Build State ---
        req_state = info.get("user_request", None)

        # --- Action Selection ---
        action, missing, transition = select_action(agent, req_state, env, net_adapter)

        # --- Environment Step ---
        _, reward, done, info = env.step(action, req_state, net_adapter)

        # --- Store Transition & Train ---
        nxt_req = info["user_request"]

        weight = 0.7
        reward_0 = info["reward_layer_0"]
        reward_1 = info["reward_layer_1"]

        reward = reward_0 + reward_1

        queued_update = False

        if missing[0] == 1 or transition[0] is not None:
            next_state_base = net_adapter.build_observation(0, nxt_req["video"])

            agent.remember(
                transition[0]['state'],
                transition[0]['action'],
                reward,
                next_state_base,
                done
            )
            queued_update = True

        for i in range(len(missing) - 1):
            if missing[i + 1] == 1 and transition[i + 1] is not None:
                next_state_enh = net_adapter.build_observation(i + 1, nxt_req["video"], nxt_req["viewport"][i])
                agent.remember(
                    transition[i + 1]['state'],
                    transition[i + 1]['action'],
                    reward,
                    next_state_enh,
                    done
                )
                queued_update = True

        if queued_update:
            train_metrics = agent.train_step()

            if train_metrics is not None:
                save_step_metrics(
                    metrics_dir=metrics_dir,
                    episode=episode,
                    episode_step=step,
                    global_step=global_step,
                    reward=reward,
                    agent=agent,
                    train_metrics=train_metrics,
                )

        delta_r, bs_hits, bs_miss, e_hits, e_miss = update_metrics(info, reward)
        total_reward += delta_r
        cache_hits += bs_hits + e_hits
        cache_misses += bs_miss + e_miss
        base_hits += bs_hits
        base_misses += bs_miss
        enh_hits += e_hits
        enh_misses += e_miss
        psnr_sum += info.get("psnr", 0.0)

        if done:
            break

        debugger.log('cache_hits', bs_hits + e_hits)
        debugger.log('cache_misses', bs_miss + e_miss)

    return total_reward, cache_hits, cache_misses, base_hits, base_misses, enh_hits, enh_misses, global_step, psnr_sum / (step + 1)


def train(cfg):
    env = builders.build_environment(cfg)

    agent = A2CWorker(cfg, debugger=debugger)

    feature_adapter = FeatureAdapter(cfg, env)
    net_adapter = NetworkAdapter(cfg, env, feature_adapter)

    date_dir = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M")
    debug_path = os.path.join(cfg.path_results, date_dir)
    metrics_dir = os.path.join(debug_path, "metrics")
    os.makedirs(metrics_dir, exist_ok=True)

    print(f"Starting training for {cfg.n_episodes} episodes... {date_dir}")
    print(f"Warmup Phase: {'Enabled' if cfg.has_warmup else 'Disabled'}")
    print(f"Users Session Length: {cfg.user_session_length}")
    print(agent)

    global_log_path = os.path.join(cfg.path_results, "global.log")
    with open(global_log_path, "a", encoding="utf-8") as f:
        f.write(f"{pd.Timestamp.now().isoformat()} | {agent}\n")
        f.write("-" * 50 + "\n")

    global_step = 0

    for episode in range(cfg.n_episodes):
        total_reward, hits, misses, bs_hits, bs_miss, enh_hits, enh_miss, global_step, psnr_rate = run_episode(
            episode, env, agent, net_adapter, cfg, metrics_dir, global_step
        )

        agent.update_epsilon()

        save_episode_metrics(
            metrics_dir=metrics_dir,
            ep=episode,
            total_reward=total_reward,
            cache_hits=hits,
            cache_misses=misses,
            agent=agent,
        )

        network_params = {"episode": episode, "networks": {}}

        for net_name in ("q_network", "policy_net", "actor", "critic", "model", "network"):
            net = getattr(agent, net_name, None)
            if net is not None and hasattr(net, "state_dict"):
                network_params["networks"][net_name] = {
                    k: v.detach().cpu().tolist()
                    for k, v in net.state_dict().items()
                }

        if network_params["networks"]:
            json_path = os.path.join(metrics_dir, f"network_params_ep{episode}.json")
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump(network_params, f, indent=2)

        debugger.log('lr', agent.scheduler.get_last_lr()[0])
        debugger.log('epsilon', agent.epsilon)

        debugger.save_results(filepath=f"{debug_path}/debug_ep{episode}")
        debugger.clear()

        print(
            f"Episode {episode} | R: {int(total_reward)} | "
            f"HR: {hits / (hits + misses + 1e-9):.2f} | "
            f"BHR: {bs_hits / (bs_hits + bs_miss + 1e-9):.2f} | "
            f"EHR: {enh_hits / (enh_hits + enh_miss + 1e-9):.2f} | "
            f"PSNR: {psnr_rate:.2f} | "
            f"Time: {pd.Timestamp.now().strftime('%H:%M:%S')}"
        )
        print("-" * 50)


if __name__ == "__main__":
    train(cfg)

Entropy floor: 0.5 | Boltzmann tau: 5.0
Starting training for 400 episodes... 2026-04-08_12-35
Warmup Phase: Disabled
Users Session Length: 60
A2CWorker(ActorLR=0.005000, CriticLR=0.001000)
BatchSize=256, Gamma=0.9
BufferSize=2000, GAE_lambda=0.95, Entropy_beta=0.01
Advantage_clip=5.0, Gradient_clip_norm=0.5
hiddens=(512, 1024, 512), Action Dim=251, State Dim=502


C:\Users\es25591\AppData\Local\Temp\ipykernel_23408\4094479284.py:34: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  if float(entropy_base) < entropy_floor:


Episode 0 | R: 207462 | HR: 0.44 | BHR: 0.51 | EHR: 0.24 | PSNR: 32.86 | Time: 12:37:16
--------------------------------------------------
Episode 1 | R: 176125 | HR: 0.37 | BHR: 0.43 | EHR: 0.20 | PSNR: 32.47 | Time: 12:39:00
--------------------------------------------------
Episode 2 | R: 177212 | HR: 0.38 | BHR: 0.43 | EHR: 0.21 | PSNR: 32.49 | Time: 12:41:45
--------------------------------------------------
Episode 3 | R: 198607 | HR: 0.42 | BHR: 0.48 | EHR: 0.24 | PSNR: 32.86 | Time: 12:45:05
--------------------------------------------------
Episode 4 | R: 177622 | HR: 0.38 | BHR: 0.43 | EHR: 0.21 | PSNR: 32.49 | Time: 12:48:39
--------------------------------------------------
Episode 5 | R: 204690 | HR: 0.43 | BHR: 0.49 | EHR: 0.25 | PSNR: 32.95 | Time: 12:51:49
--------------------------------------------------
Episode 6 | R: 177165 | HR: 0.38 | BHR: 0.43 | EHR: 0.20 | PSNR: 32.46 | Time: 12:55:20
--------------------------------------------------
Episode 7 | R: 163952 | HR:

KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn.functional as F
from torch.distributions import Categorical
import math

def boltzmann_exploration(q_values, tau=1.0):
    """
    Selects an action using Boltzmann (Softmax) Exploration.
    
    Args:
        q_values: Tensor of raw Q-values. Example: torch.tensor([[3.0, 2.5, 0.5]])
        tau: The Temperature parameter. 
             High = More random, Low = More greedy.
    """
    # Step 1: Scale the Q-values by the temperature
    scaled_q_values = q_values / tau
    
    # Step 2: Convert to probabilities using Softmax
    action_probs = F.softmax(scaled_q_values, dim=-1)
    
    # Step 3: Create a categorical distribution and sample an action
    # This automatically rolls a weighted die based on the probabilities
    dist = Categorical(action_probs)
    action = dist.sample().item()
    
    return action, action_probs.squeeze().tolist()

# --- Example Scenarios ---
# Let's say Action 0 is slightly better than Action 1, and much better than Action 2.
q_current = torch.tensor([[3.0, 2.5, 0.5]])

# Scenario 1: High Temperature (tau = 5.0) -> "Exploration Injection"
# The agent is confused/exploring. Even the worst action has a decent chance.
act_high, probs_high = boltzmann_exploration(q_current, tau=2.0)
print(f"High Temp (tau=5)  -> Probs: {[f'{p:.3f}' for p in probs_high]}")
# Output: ['0.375', '0.339', '0.286']

# Scenario 2: Normal Temperature (tau = 1.0)
act_mid, probs_mid = boltzmann_exploration(q_current, tau=1.0)
print(f"Normal Temp (tau=1)-> Probs: {[f'{p:.3f}' for p in probs_mid]}")
# Output: ['0.613', '0.372', '0.050']

# Scenario 3: Low Temperature (tau = 0.2) -> Greedy Exploitation
# The agent is highly confident. The slightly worse action is almost ignored.
act_low, probs_low = boltzmann_exploration(q_current, tau=0.2)
print(f"Low Temp (tau=0.2) -> Probs: {[f'{p:.3f}' for p in probs_low]}")
# Output: ['0.924', '0.076', '0.000']

# Calculate entropy for each scenario

def calculate_entropy(probs):
    """Calculate Shannon entropy of a probability distribution."""
    return -sum(p * math.log(p + 1e-10) for p in probs)

entropy_high = calculate_entropy(probs_high)
entropy_mid = calculate_entropy(probs_mid)
entropy_low = calculate_entropy(probs_low)

print(f"\nEntropy (High Temp): {entropy_high:.4f}")
print(f"Entropy (Normal Temp): {entropy_mid:.4f}")
print(f"Entropy (Low Temp): {entropy_low:.4f}")

High Temp (tau=5)  -> Probs: ['0.484', '0.377', '0.139']
Normal Temp (tau=1)-> Probs: ['0.592', '0.359', '0.049']
Low Temp (tau=0.2) -> Probs: ['0.924', '0.076', '0.000']

Entropy (High Temp): 0.9930
Entropy (Normal Temp): 0.8250
Entropy (Low Temp): 0.2686
